# 8. Magi's fingerprints: the same character across pages

After the VLMs did not help much I went back to magi. Its biggest problem: it only groups characters
inside one page, so on the next page the same girl gets a new name.

Inside magiv2 there is a crop embedder: it turns a picture of a character into 768 numbers.
Two pictures of the same character get similar numbers -> a "fingerprint".

Plan:
- every magi group gets one fingerprint (average of its boxes)
- keep joining the two most alike groups (also across pages) while they are more alike than T
- every line gets the speaker of its group, lines magi linked to nobody get the previous speaker
- keep only the CAP most talkative speakers

Kaggle: GPU T4, internet on. Inputs: task data and `magi_output.json` (notebook 1).

In [1]:
%pip install -q "transformers==4.49.0" sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.0 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import sys
import json
import math
import random
import numpy as np
from PIL import Image
from collections import Counter
from functools import lru_cache
import torch
from transformers import AutoModel

In [3]:
def find(name):
    for top in ["/kaggle/working", "/kaggle/input", ".."]:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files:
                return root + "/" + name
    return None

DATA = os.path.dirname(find("sequences.json"))
MAGI = find("magi_output.json")
print(DATA)
print(MAGI)
OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "../data"

sys.path.insert(0, DATA)
import score
from score import load_jsonl, evaluate, tokens, align_tokens

magi = json.load(open(MAGI))
gold = load_jsonl(DATA + "/development/labels.jsonl", reference=True)
images = {}
for s in json.load(open(DATA + "/sequences.json")):
    images[s["sequence_id"]] = [DATA + "/" + p for p in s["images"]]
dev = sorted(gold.keys())
test = [s for s in magi if magi[s]["split"] == "test"]

/kaggle/input/datasets/arpitparashar06/manga-task/manga-task-upload/manga-task/dataset
/kaggle/input/datasets/arpitparashar06/manga-task/magi_output.json


In [4]:
# speed up: the scorer lines up our words with the gold words every time.
# while tuning only the speaker names change, not the text, so remember the line-up.
original_align = score.align_tokens
saved = {}

def remembered_align(g, p):
    key = (tuple(t[0] for t in g), tuple(t[0] for t in p))
    if key not in saved:
        saved[key] = [(a[1], b[1], w) for a, b, w in original_align([(g[i][0], i) for i in range(len(g))],
                                                                    [(p[j][0], j) for j in range(len(p))])]
    return [(g[i], p[j], w) for i, j, w in saved[key]]

score.align_tokens = remembered_align
score.distance = lru_cache(maxsize=None)(score.distance)     # same for the page text comparison

## fingerprints

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModel.from_pretrained("ragavsachdeva/magiv2", trust_remote_code=True).to(device).eval()
embedder = model.crop_embedding_model
embedder.config.mask_ratio = 0.0          # it's a ViT-MAE, by default it hides 75% of the picture

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def crop(img, b):
    # same box fixing as magi: whole pixels, inside the page, at least 10 px
    H, W = img.shape[0], img.shape[1]
    x1 = max(0, min(int(min(b[0], b[2])), W))
    x2 = max(0, min(int(max(b[0], b[2])), W))
    y1 = max(0, min(int(min(b[1], b[3])), H))
    y2 = max(0, min(int(max(b[1], b[3])), H))
    if x2 - x1 < 10:
        if W - x1 > 10:
            x2 = x1 + 10
        else:
            x1 = x2 - 10
    if y2 - y1 < 10:
        if H - y1 > 10:
            y2 = y1 + 10
        else:
            y1 = y2 - 10
    return img[y1:y2, x1:x2]

def fingerprints(img, boxes):
    batch = []
    for b in boxes:
        c = Image.fromarray(crop(img, b)).resize((224, 224), Image.BILINEAR)
        x = (np.asarray(c).astype(np.float32) / 255.0 - MEAN) / STD
        batch.append(x.transpose(2, 0, 1))
    with torch.no_grad():
        out = embedder(pixel_values=torch.tensor(np.stack(batch)).to(device)).last_hidden_state[:, 0]
    return out.float().cpu().numpy()

config.json: 0.00B [00:00, ?B/s]

configuration_magiv2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ragavsachdeva/magiv2:
- configuration_magiv2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modelling_magiv2.py: 0.00B [00:00, ?B/s]

processing_magiv2.py: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.06G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.49.0"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

In [6]:
emb = {}
for sid in magi:
    for p in range(3):
        boxes = magi[sid]["pages"][p]["char_boxes"]
        if len(boxes) > 0:
            img = np.array(Image.open(images[sid][p]).convert("L").convert("RGB"))
            emb[sid + "|" + str(p)] = fingerprints(img, boxes).round(4).tolist()
json.dump(emb, open(OUT + "/char_embeddings.json", "w"))
print("fingerprints for", len(emb), "pages")

fingerprints for 285 pages


In [7]:
def read_sequence(sid, emb=None):
    # lines = every story line in reading order: [page, text, group]
    # group = (page, magi group) or None if magi linked it to nobody
    lines = []
    groups = {}
    for p in range(3):
        r = magi[sid]["pages"][p]
        link = {a: b for a, b in r["links"]}
        for i in range(len(r["ocr"])):
            if r["keep"][i] == True and r["ocr"][i].strip() != "":
                group = None
                c = link.get(i, -1)
                if 0 <= c < len(r["clusters"]):
                    group = (p, r["clusters"][c])
                    if group not in groups:
                        groups[group] = {"lines": []}
                    groups[group]["lines"].append(len(lines))
                lines.append([p, r["ocr"][i].strip(), group])

        # every group gets one fingerprint: the average of its boxes
        key = sid + "|" + str(p)
        if emb is None or key not in emb or len(emb[key]) == 0:
            continue
        E = np.array(emb[key], dtype=np.float32)
        E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)
        for (gp, g) in groups:
            if gp != p:
                continue
            rows = [c for c in range(min(len(r["clusters"]), len(E))) if r["clusters"][c] == g]
            if len(rows) == 0:
                continue
            v = E[rows].mean(0)
            groups[(p, g)]["vec"] = v / (np.linalg.norm(v) + 1e-8)
    return lines, groups

In [8]:
# does it work? same magi group = same person for sure. how alike are fingerprints
# of the same group vs different groups on one page?
same, diff = [], []
for sid in dev:
    for p in range(3):
        key = sid + "|" + str(p)
        if key not in emb:
            continue
        E = np.array(emb[key])
        E = E / np.linalg.norm(E, axis=1, keepdims=True)
        cl = magi[sid]["pages"][p]["clusters"]
        for a in range(len(E)):
            for b in range(a + 1, len(E)):
                (same if cl[a] == cl[b] else diff).append(float(E[a] @ E[b]))
print("same group:      %.2f" % np.mean(same))
print("different group: %.2f" % np.mean(diff))

same group:      0.85
different group: 0.48


## joining groups into characters

In [9]:
def merge_groups(groups, T):
    # start with every magi group alone, keep joining the two most alike while they are above T
    keys = sorted([k for k in groups if "vec" in groups[k]])
    clusters = [[k] for k in keys]
    while True:
        best = None
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                s = float(np.mean([groups[a]["vec"] @ groups[b]["vec"] for a in clusters[i] for b in clusters[j]]))
                if s > T and (best is None or s > best[0]):
                    best = (s, i, j)
        if best is None:
            break
        clusters[best[1]] = clusters[best[1]] + clusters[best[2]]
        del clusters[best[2]]
    person = {}
    for n in range(len(clusters)):
        for k in clusters[n]:
            person[k] = "c" + str(n)
    return person

def finish(lines, person, CAP):
    rows = []
    for page, text, g in lines:
        spk = None
        if g is not None:
            spk = person[g] if g in person else "p" + str(g[0]) + "_g" + str(g[1])
        rows.append([page, text, spk])

    # magi linked it to nobody -> previous speaker (or the next one at the start)
    for i in range(len(rows)):
        if rows[i][2] is None:
            spk = None
            for x in range(i - 1, -1, -1):
                if rows[x][2] is not None:
                    spk = rows[x][2]
                    break
            if spk is None:
                for x in range(i + 1, len(rows)):
                    if rows[x][2] is not None:
                        spk = rows[x][2]
                        break
            rows[i][2] = spk if spk is not None else "NARRATION"

    # keep the CAP most talkative, the rest -> previous kept speaker
    count = Counter(r[2] for r in rows if r[2] != "NARRATION")
    keep = [name for name, n in count.most_common(CAP)]
    last = None
    for r in rows:
        if r[2] != "NARRATION" and r[2] not in keep:
            r[2] = last if last is not None else keep[0]
        if r[2] != "NARRATION":
            last = r[2]

    names = {}
    pages = [[], [], []]
    for page, text, spk in rows:
        if spk != "NARRATION":
            if spk not in names:
                names[spk] = "character" + str(len(names) + 1)
            spk = names[spk]
        pages[page].append({"speaker": spk, "text": text})
    return pages

def predict(sid, T, CAP):
    lines, groups = read_sequence(sid, emb)
    return finish(lines, merge_groups(groups, T), CAP)

In [10]:
# a kaguya-sama sequence: magi alone vs with fingerprints
sid = "seq_b251c8e1ba8b91f8"
lines, groups = read_sequence(sid, emb)
pred = predict(sid, 0.75, 3)
n = 0
for p in range(3):
    for x in pred[p]:
        g = lines[n][2]
        print("page", p, " magi: %-8s  now: %-11s %s" % ("-" if g is None else "p%d_g%d" % g, x["speaker"], x["text"][:50]))
        n += 1

page 0  magi: p0_g1     now: character1  Now... how will you react...?
page 0  magi: p0_g2     now: character1  Shinomiya thought for an instant...
page 0  magi: p0_g1     now: character1  But what went through her mind was no ordinary pro
page 0  magi: p0_g0     now: character2  He's attacked me...
page 0  magi: p0_g0     now: character2  By revealing that he wants to see the movie, he's 
page 0  magi: p0_g0     now: character2  A fine counter- attack...
page 0  magi: p0_g1     now: character1  I have the option of refusing him,
page 0  magi: p0_g1     now: character1  But that would render all I've done up to this poi
page 0  magi: p0_g1     now: character1  All I've done...
page 1  magi: -         now: character1  Carefully preparing a fake prize...
page 1  magi: p1_g1     now: character1  Slipping it into Fuiura's mailbox...
page 1  magi: p1_g2     now: character1  and finding a movie on a weekend on one of his rar
page 1  magi: p1_g0     now: character2  If I refuse him here, with

## score
before: magi as it is (speaker = magi group of that page, no link = NARRATION), like notebook 1

In [11]:
def avg(scores, ids):
    # same as the scorer: mean over the sequences that have a score
    vals = [scores[s] for s in ids if scores[s] is not None]
    return sum(vals) / len(vals)

def score_each(predict_fn):
    # balanced_joint_f1 of every dev sequence on its own
    return {sid: evaluate({sid: gold[sid]}, {sid: predict_fn(sid)})["macro"]["balanced_joint_f1"] for sid in dev}

def cross_validate(grid, seed):
    # pick the setting on 4/5 of the sequences, score it on the other 1/5
    ids = list(dev)
    random.Random(seed).shuffle(ids)
    total = 0
    for k in range(5):
        held = ids[k::5]
        fit = [s for s in ids if s not in held]
        best = max(grid, key=lambda st: avg(grid[st], fit))
        total += avg(grid[best], held) / 5
    return total

def magi_as_is(sid):
    lines, groups = read_sequence(sid)
    pages = [[], [], []]
    for p, text, g in lines:
        pages[p].append({"speaker": "NARRATION" if g is None else "group" + str(g[1]), "text": text})
    return pages

print("magi as it is: %.3f" % avg(score_each(magi_as_is), dev))

magi as it is: 0.342


In [12]:
# every T and CAP, every dev sequence
grid = {}
for T in [0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85]:
    for CAP in [99, 4, 3]:
        grid[(T, CAP)] = score_each(lambda sid: predict(sid, T, CAP))
        print("T=%.2f CAP=%2d  %.3f" % (T, CAP, avg(grid[(T, CAP)], dev)))
T, CAP = max(grid, key=lambda k: avg(grid[k], dev))
print("best on all 80:", T, CAP, "%.3f (optimistic, picked and scored on the same data)" % avg(grid[(T, CAP)], dev))

T=0.55 CAP=99  0.385
T=0.55 CAP= 4  0.385
T=0.55 CAP= 3  0.382
T=0.60 CAP=99  0.414
T=0.60 CAP= 4  0.413
T=0.60 CAP= 3  0.411
T=0.65 CAP=99  0.448
T=0.65 CAP= 4  0.449
T=0.65 CAP= 3  0.449
T=0.70 CAP=99  0.471
T=0.70 CAP= 4  0.477
T=0.70 CAP= 3  0.477
T=0.75 CAP=99  0.467
T=0.75 CAP= 4  0.473
T=0.75 CAP= 3  0.482
T=0.80 CAP=99  0.458
T=0.80 CAP= 4  0.468
T=0.80 CAP= 3  0.477
T=0.85 CAP=99  0.428
T=0.85 CAP= 4  0.450
T=0.85 CAP= 3  0.459
best on all 80: 0.75 3 0.482 (optimistic, picked and scored on the same data)


In [13]:
# honest number: 5-fold cross-validation.
# with only 16 sequences per fold the split itself moves the score by ~0.02,
# so i repeat it with 200 different shuffles and take the mean
cv = np.array([cross_validate(grid, seed) for seed in range(200)])
print("5-fold CV, mean of 200 splits: %.3f  (5%%-95%% of the splits: %.3f-%.3f)" % (cv.mean(), np.percentile(cv, 5), np.percentile(cv, 95)))

5-fold CV, mean of 200 splits: 0.468  (5%-95% of the splits: 0.456-0.477)


In [14]:
f = open(OUT + "/fingerprints_dev.jsonl", "w")
for sid in dev:
    f.write(json.dumps({"sequence_id": sid, "pages": predict(sid, T, CAP)}) + "\n")
f.close()

!python "{DATA}/score.py" --references "{DATA}/development/labels.jsonl" --predictions "{OUT}/fingerprints_dev.jsonl" --output "{OUT}/fingerprints_dev_scores.json"

{
  "text_order_score": 0.9293819876464963,
  "balanced_joint_f1": 0.482136018620755,
  "speaker_accuracy_on_matched": 0.6847864073358146,
  "joint_f1": 0.6468518062796711,
  "matched_token_coverage": 0.9627321856604695
}
